# WDMamba Phase 2.1: Corrective RGB refinement

This corrective phase addresses Phase 2 underfitting. It evaluates three residual multi-scale U-Net configurations, uses the complete B0 loss family, EMA weights, a longer training budget, and the exact Phase 1 evaluation protocol. It still excludes ASM and explicit wavelet guidance.

Gate G2.1 requires ΔPSNR >= -0.3 dB and at least 1.25x speedup against Phase 1, a clear quality improvement over the failed Phase 2 B1, protocol equality, and manual artifact review.

## 1. Clone and configure

The notebook runs a 10k-iteration search for three configurations, then trains the selected configuration for 50k iterations on the complete RESIDE-6K train split. Evaluation batch size and `MAX_SIDE` are inherited from Phase 1.

In [ ]:
from pathlib import Path
import json, platform, subprocess, sys

REPO_URL = "https://github.com/Darkin72/WDMamba.git"
REPO_ROOT = Path("/content/WDMamba_phase21")
WORK_ROOT = Path("/content/wdmamba_phase21")
OUTPUT_ROOT = WORK_ROOT / "results"
ENV_ROOT = Path("/content/wdmamba_phase21_python")
PYTHON = ENV_ROOT / "bin/python"
if platform.system() != "Linux" or not Path("/content").is_dir():
    raise RuntimeError("This notebook is intended for Google Colab/Linux.")

def run_command(args, cwd=None, log_path=None):
    args = [str(x) for x in args]; print("$", " ".join(args), flush=True)
    log_path = Path(log_path or WORK_ROOT / "logs" / "command.log"); log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True); log.write(line); log.flush()
        rc = process.wait()
    if rc: raise RuntimeError(f"Command failed ({rc}); full log: {log_path}")

if not REPO_ROOT.exists(): run_command(["git", "clone", "--depth", "1", REPO_URL, REPO_ROOT])
run_command(["git", "log", "-1", "--oneline"], cwd=REPO_ROOT)

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
PATCH_SIZE = 256
MAX_SIDE = 0
SEARCH_ITERS = 10000
FINAL_ITERS = 50000
SEARCH_DEV_LIMIT = 512
TEST_LIMIT = 0
NUM_WORKERS = 2
WARMUP = 10
TIMING_REPEATS = 30
SEED = 666
ARTIFACTS_REVIEWED = False
P1_BASELINE_ACCEPTED = False
P1_SUMMARY_PATH = ""
P2_SUMMARY_PATH = ""
KAGGLE_DATASET = "kmljts/reside-6k"
SEARCH_CONFIGS = [
    {"name": "B1C-24", "base": 24, "depth": 2, "detail_blocks": 2},
    {"name": "B1C-32", "base": 32, "depth": 2, "detail_blocks": 3},
    {"name": "B1C-40", "base": 40, "depth": 3, "detail_blocks": 3},
]
CONFIG = {"repo_root": str(REPO_ROOT), "work_root": str(WORK_ROOT), "output_root": str(OUTPUT_ROOT),
          "train_batch_size": TRAIN_BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE,
          "patch_size": PATCH_SIZE, "max_side": MAX_SIDE, "search_iters": SEARCH_ITERS,
          "final_iters": FINAL_ITERS, "search_dev_limit": SEARCH_DEV_LIMIT, "test_limit": TEST_LIMIT,
          "num_workers": NUM_WORKERS, "warmup": WARMUP, "timing_repeats": TIMING_REPEATS,
          "seed": SEED, "artifacts_reviewed": ARTIFACTS_REVIEWED,
          "p1_baseline_accepted": P1_BASELINE_ACCEPTED, "p1_summary_path": P1_SUMMARY_PATH,
          "p2_summary_path": P2_SUMMARY_PATH, "kaggle_dataset": KAGGLE_DATASET,
          "search_configs": SEARCH_CONFIGS}
WORK_ROOT.mkdir(parents=True, exist_ok=True); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = WORK_ROOT / "config.json"; CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print(json.dumps(CONFIG, indent=2))

### Upload Phase 1 and Phase 2 summaries

Upload exactly two CSV files: the full Phase 1 `summary.csv` and the failed Phase 2 `test_summary.csv`. Phase 1 defines the evaluation protocol and official quality/speed target; Phase 2 measures corrective improvement.

In [ ]:
from google.colab import files
import pandas as pd
uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if len(csv_names) != 2:
    raise ValueError("Upload exactly two CSV files: Phase 1 summary.csv and Phase 2 test_summary.csv.")
frames = {name: pd.read_csv(Path(name)) for name in csv_names}
p1_name = next((name for name, frame in frames.items() if "original_forward_ms_per_image" in frame.columns), None)
p2_name = next((name for name, frame in frames.items() if "latency_ms_per_image" in frame.columns), None)
if p1_name is None or p2_name is None: raise ValueError("Could not identify the Phase 1 and Phase 2 summaries.")
p1_target = WORK_ROOT / "P1_full_summary.csv"; p1_target.write_bytes(uploaded[p1_name]); P1_SUMMARY_PATH = str(p1_target)
p2_target = WORK_ROOT / "P2_failed_summary.csv"; p2_target.write_bytes(uploaded[p2_name]); P2_SUMMARY_PATH = str(p2_target)
p1 = frames[p1_name].iloc[0]; EVAL_BATCH_SIZE = int(p1["configured_batch_size"]); MAX_SIDE = int(p1["max_side"])
CONFIG.update({"p1_summary_path": P1_SUMMARY_PATH, "p2_summary_path": P2_SUMMARY_PATH,
               "eval_batch_size": EVAL_BATCH_SIZE, "max_side": MAX_SIDE})
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print("Inherited protocol: eval batch", EVAL_BATCH_SIZE, "MAX_SIDE", MAX_SIDE)

## 2. Install the isolated environment

In [ ]:
run_command([sys.executable, "-m", "pip", "install", "-q", "uv"])
run_command([sys.executable, "-m", "uv", "python", "install", "3.11"])
if not PYTHON.is_file(): run_command([sys.executable, "-m", "uv", "venv", "--python", "3.11", ENV_ROOT])
run_command([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON,
             "--index-url", "https://download.pytorch.org/whl/cu128", "torch==2.8.0", "torchvision==0.23.0"])
constraints = ENV_ROOT / "constraints.txt"; constraints.write_text("torch==2.8.0\ntorchvision==0.23.0\nnumpy<2\n", encoding="utf-8")
run_command([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON, "--constraint", constraints,
             "numpy<2", "opencv-python-headless<4.12", "scipy", "scikit-image", "pandas", "matplotlib", "tqdm", "kagglehub"])

## 3. Download and resolve RESIDE-6K train/test pairs

In [ ]:
LOCAL_DATA_ROOT = ""
CONFIG["local_data_root"] = LOCAL_DATA_ROOT; CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
def run_python(label, source):
    script=WORK_ROOT/f"{label}.py"; script.write_text(source.strip()+"\n",encoding="utf-8")
    run_command([PYTHON,"-u",script,CONFIG_PATH],cwd=REPO_ROOT,log_path=WORK_ROOT/"logs"/f"{label}.log")
run_python("prepare_reside6k", r"""
import json,sys
from pathlib import Path
import kagglehub
cfg=json.loads(Path(sys.argv[1]).read_text(encoding="utf-8")); base=Path(cfg["local_data_root"]) if cfg.get("local_data_root") else Path(kagglehub.dataset_download(cfg["kaggle_dataset"])); ext={".png",".jpg",".jpeg",".bmp",".tif",".tiff"}
def pair(path):
    d={p.name.lower():p for p in path.iterdir() if p.is_dir()}; gt=next((d[k] for k in ("gt","clear","clean","groundtruth") if k in d),None); lq=next((d[k] for k in ("hazy","haze","input","inputs") if k in d),None)
    return (gt,lq) if gt and lq and any(p.suffix.lower() in ext for p in gt.iterdir() if p.is_file()) else None
rows=[]
for path in [base,*[p for p in base.rglob("*") if p.is_dir()]]:
    found=pair(path)
    if found: rows.append((path,*found))
if not rows: raise FileNotFoundError(f"No pairs below {base}")
def score(row,key): return (0 if key in "/".join(p.name.lower() for p in row) else 1,len(str(row[0])))
train=sorted(rows,key=lambda x:score(x,"train"))[0]; test=sorted(rows,key=lambda x:score(x,"test"))[0]
if train[0]==test[0] and len(rows)>1: test=sorted(rows,key=lambda x:score(x,"test"))[1]
data={"train_clear":str(train[1].resolve()),"train_hazy":str(train[2].resolve()),"test_clear":str(test[1].resolve()),"test_hazy":str(test[2].resolve())}; Path(cfg["work_root"],"inputs.json").write_text(json.dumps(data,indent=2),encoding="utf-8"); print(json.dumps(data,indent=2))
""")

## 4. Search and train the corrective B1C model

The search uses all three configurations. The selected candidate is the fastest model within 0.15 dB of the best development PSNR, then it is retrained on all 6,000 training pairs.

In [ ]:
import ast, textwrap
TRAINER_SOURCE = textwrap.dedent('\nimport copy,json,math,random,sys,time\nfrom pathlib import Path\nimport cv2,numpy as np,pandas as pd,torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom skimage.metrics import structural_similarity\nfrom torch.utils.data import Dataset,DataLoader\nfrom torchvision.models import vgg19,VGG19_Weights\nfrom tqdm import tqdm\nCFG=json.loads(Path(sys.argv[1]).read_text(encoding="utf-8")); INPUTS=json.loads((Path(CFG["work_root"])/"inputs.json").read_text()); OUT=Path(CFG["output_root"]); DEVICE=torch.device("cuda"); assert torch.cuda.is_available(); random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"]); torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False\nEXT={".png",".jpg",".jpeg",".bmp",".tif",".tiff"}\ndef images(root): return sorted(p for p in Path(root).iterdir() if p.is_file() and p.suffix.lower() in EXT)\ndef pairs(gt,lq,limit=0):\n    clear={p.stem:p for p in images(gt)}; rows=[]\n    for p in images(lq):\n        g=clear.get(p.stem,clear.get(p.stem.split("_")[0]))\n        if g: rows.append((p.stem,p,g))\n    return rows[:limit] if limit>0 else rows\nclass Data(Dataset):\n    def __init__(self,rows,train): self.rows,self.train=rows,train\n    def __len__(self): return len(self.rows)\n    def __getitem__(self,i):\n        name,xp,yp=self.rows[i]; x=cv2.cvtColor(cv2.imread(str(xp)),cv2.COLOR_BGR2RGB); y=cv2.cvtColor(cv2.imread(str(yp)),cv2.COLOR_BGR2RGB)\n        if x.shape[:2]!=y.shape[:2]: x=cv2.resize(x,(y.shape[1],y.shape[0]),interpolation=cv2.INTER_AREA)\n        if self.train:\n            s=min(CFG["patch_size"],*y.shape[:2]); top=random.randint(0,y.shape[0]-s); left=random.randint(0,y.shape[1]-s); x,y=x[top:top+s,left:left+s],y[top:top+s,left:left+s]\n            if random.random()<.5: x,y=x[:,::-1],y[:,::-1]\n            k=random.randint(0,3); x,y=np.rot90(x,k).copy(),np.rot90(y,k).copy()\n        elif CFG["max_side"]>0 and max(y.shape[:2])>CFG["max_side"]:\n            scale=CFG["max_side"]/max(y.shape[:2]); size=(round(y.shape[1]*scale),round(y.shape[0]*scale)); x=cv2.resize(x,size,interpolation=cv2.INTER_AREA); y=cv2.resize(y,size,interpolation=cv2.INTER_AREA)\n        return torch.from_numpy(x.copy()).permute(2,0,1).float()/255,torch.from_numpy(y.copy()).permute(2,0,1).float()/255,name\ndef collate(batch):\n    h=math.ceil(max(x[0].shape[-2] for x in batch)/4)*4; w=math.ceil(max(x[0].shape[-1] for x in batch)/4)*4; xs=[];ys=[];names=[];shapes=[]\n    for x,y,n in batch: shapes.append(y.shape[-2:]); xs.append(F.pad(x,(0,w-x.shape[-1],0,h-x.shape[-2]),mode="replicate")); ys.append(F.pad(y,(0,w-y.shape[-1],0,h-y.shape[-2]),mode="replicate")); names.append(n)\n    return torch.stack(xs),torch.stack(ys),names,shapes\nclass ResBlock(nn.Module):\n    def __init__(self,c): super().__init__(); self.net=nn.Sequential(nn.Conv2d(c,c,3,1,1),nn.GELU(),nn.Conv2d(c,c,3,1,1)); self.scale=nn.Parameter(torch.tensor(.1))\n    def forward(self,x): return x+self.scale*self.net(x)\nclass Stage(nn.Module):\n    def __init__(self,c,n): super().__init__(); self.net=nn.Sequential(*[ResBlock(c) for _ in range(n)])\n    def forward(self,x): return self.net(x)\nclass CorrectiveUNet(nn.Module):\n    def __init__(self,base,depth,detail_blocks):\n        super().__init__(); self.stem=nn.Conv2d(3,base,3,1,1); self.e1=Stage(base,depth); self.d1=nn.Conv2d(base,base*2,4,2,1); self.e2=Stage(base*2,depth); self.d2=nn.Conv2d(base*2,base*4,4,2,1); self.mid=Stage(base*4,depth+1); self.u2=nn.Conv2d(base*4,base*2,3,1,1); self.c2=nn.Conv2d(base*4,base*2,1); self.dec2=Stage(base*2,depth); self.u1=nn.Conv2d(base*2,base,3,1,1); self.c1=nn.Conv2d(base*2,base,1); self.dec1=Stage(base,depth); self.detail=Stage(base,detail_blocks); self.tail=nn.Conv2d(base,3,3,1,1)\n    def forward(self,x):\n        a=self.e1(self.stem(x)); b=self.e2(self.d1(a)); m=self.mid(self.d2(b)); u=F.interpolate(m,size=b.shape[-2:],mode="bilinear",align_corners=False); u=self.dec2(self.c2(torch.cat((self.u2(u),b),1))); u=F.interpolate(u,size=a.shape[-2:],mode="bilinear",align_corners=False); u=self.detail(self.dec1(self.c1(torch.cat((self.u1(u),a),1)))); return (x+self.tail(u)).clamp(0,1)\nclass Contrast(nn.Module):\n    def __init__(self):\n        super().__init__(); self.f=vgg19(weights=VGG19_Weights.IMAGENET1K_V1).features[:30].eval(); self.pick={1,6,11,20,29}; self.w=[1/32,1/16,1/8,1/4,1]\n        for p in self.parameters(): p.requires_grad=False\n    def enc(self,x):\n        out=[]\n        for i,l in enumerate(self.f):\n            x=l(x)\n            if i in self.pick: out.append(x)\n        return out\n    def forward(self,a,p,n): return sum(w*F.l1_loss(x,y.detach())/(F.l1_loss(x,z.detach())+1e-7) for w,x,y,z in zip(self.w,self.enc(a),self.enc(p),self.enc(n)))\nCON=Contrast().to(DEVICE)\ndef ll(x): return (x[:,:,0::2,0::2]+x[:,:,1::2,0::2]+x[:,:,0::2,1::2]+x[:,:,1::2,1::2])/2\ndef ssim_loss(x,y):\n    mx=F.avg_pool2d(x,3,1,1); my=F.avg_pool2d(y,3,1,1); vx=F.avg_pool2d(x*x,3,1,1)-mx*mx; vy=F.avg_pool2d(y*y,3,1,1)-my*my; vxy=F.avg_pool2d(x*y,3,1,1)-mx*my; return 1-(((2*mx*my+.0001)*(2*vxy+.0009))/((mx*mx+my*my+.0001)*(vx+vy+.0009))).mean()\ndef loss_fn(o,y,x): return F.l1_loss(o,y)+.1*ssim_loss(o,y)+F.l1_loss(ll(o),ll(y))+.1*(torch.fft.rfft2(o).abs()-torch.fft.rfft2(y).abs()).abs().mean()+.001*CON(o,y,x)\ndef update_ema(ema,model,d=.999):\n    with torch.no_grad():\n        for a,b in zip(ema.parameters(),model.parameters()): a.mul_(d).add_(b,alpha=1-d)\ndef train(spec,rows,iters,tag):\n    model=CorrectiveUNet(spec["base"],spec["depth"],spec["detail_blocks"]).to(DEVICE); ema=copy.deepcopy(model).eval(); opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4); sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,iters,eta_min=1e-6); loader=DataLoader(Data(rows,True),batch_size=CFG["train_batch_size"],shuffle=True,num_workers=CFG["num_workers"],pin_memory=True,drop_last=True); it=iter(loader); history=[]; start=time.perf_counter()\n    model.train()\n    for step in tqdm(range(1,iters+1),desc=tag):\n        try:x,y,_=next(it)\n        except StopIteration:it=iter(loader);x,y,_=next(it)\n        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True);opt.zero_grad(set_to_none=True);o=model(x);loss=loss_fn(o,y,x);loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step();sch.step();update_ema(ema,model)\n        if step%100==0: history.append({"step":step,"loss":float(loss.detach()),"lr":sch.get_last_lr()[0]})\n    seconds=time.perf_counter()-start; ckpt=OUT/f"{tag}.pth";torch.save({"model":model.state_dict(),"ema":ema.state_dict(),"spec":spec,"iters":iters,"train_seconds":seconds},ckpt);pd.DataFrame(history).to_csv(OUT/f"{tag}_history.csv",index=False);return ema,{"train_seconds":seconds,"iters":iters,"checkpoint":str(ckpt),"final_loss":history[-1]["loss"]}\ndef evaluate(model,rows,tag,save=False):\n    model.eval();loader=DataLoader(Data(rows,False),batch_size=CFG["eval_batch_size"],shuffle=False,num_workers=CFG["num_workers"],pin_memory=True,collate_fn=collate);data=[];fails=[];gpu=0;total=0;warm=set();torch.cuda.reset_peak_memory_stats()\n    with torch.inference_mode():\n        for x,y,names,shapes in tqdm(loader,desc=tag):\n            x=x.to(DEVICE,non_blocking=True);key=tuple(x.shape)\n            if key not in warm:\n                for _ in range(CFG["warmup"]):model(x)\n                torch.cuda.synchronize();warm.add(key)\n            times=[]\n            for _ in range(CFG["timing_repeats"]):a=torch.cuda.Event(True);b=torch.cuda.Event(True);a.record();o=model(x);b.record();b.synchronize();times.append(a.elapsed_time(b))\n            gpu+=float(np.mean(times));on=o.cpu().numpy();yn=y.numpy();xn=x.cpu().numpy()\n            for j,(name,shape) in enumerate(zip(names,shapes)):\n                h,w=shape;pred=np.clip(on[j,:,:h,:w].transpose(1,2,0)*255,0,255).round().astype(np.uint8);gt=np.clip(yn[j,:,:h,:w].transpose(1,2,0)*255,0,255).round().astype(np.uint8);inp=np.clip(xn[j,:,:h,:w].transpose(1,2,0)*255,0,255).round().astype(np.uint8);mse=np.mean((pred.astype(float)-gt.astype(float))**2);psnr=99 if mse==0 else 10*np.log10(255**2/mse);ssim=structural_similarity(pred,gt,channel_axis=-1,data_range=255,gaussian_weights=True,sigma=1.5,use_sample_covariance=False);data.append({"model":tag,"image_id":name,"PSNR":psnr,"SSIM":ssim});fails.append((psnr,name,inp,pred,gt))\n            total+=len(names)\n    if save:\n        folder=OUT/"failure_cases"/tag;folder.mkdir(parents=True,exist_ok=True)\n        for _,name,inp,pred,gt in sorted(fails)[:12]:\n            for suffix,img in (("input",inp),("pred",pred),("gt",gt)):cv2.imwrite(str(folder/f"{name}_{suffix}.png"),cv2.cvtColor(img,cv2.COLOR_RGB2BGR))\n    frame=pd.DataFrame(data);summary={"model":tag,"images":len(frame),"PSNR":frame.PSNR.mean(),"SSIM":frame.SSIM.mean(),"PSNR_std":frame.PSNR.std(),"SSIM_std":frame.SSIM.std(),"latency_ms_per_image":gpu/total,"FPS":1000*total/gpu,"peak_allocated_GiB":torch.cuda.max_memory_allocated()/1024**3,"params_M":sum(p.numel() for p in model.parameters())/1e6};return summary,frame\ndef main():\n    train_rows=pairs(INPUTS["train_clear"],INPUTS["train_hazy"]);test_rows=pairs(INPUTS["test_clear"],INPUTS["test_hazy"],CFG["test_limit"]);random.shuffle(train_rows);dev=train_rows[:min(CFG["search_dev_limit"],len(train_rows))];search=[];specs={}\n    for spec in CFG["search_configs"]:\n        model,info=train(spec,dev,CFG["search_iters"],spec["name"]+"_search");row,_=evaluate(model,dev,spec["name"]+"_dev");row.update(info);search.append(row);specs[spec["name"]]=spec\n    pd.DataFrame(search).to_csv(OUT/"search_results.csv",index=False);best_psnr=max(x["PSNR"] for x in search);chosen=min((x for x in search if x["PSNR"]>=best_psnr-.15),key=lambda x:x["latency_ms_per_image"]);spec=specs[chosen["model"].split("_dev")[0]];model,info=train(spec,train_rows,CFG["final_iters"],"B1C_selected_final");ds,di=evaluate(model,dev,"B1C_selected_dev");ts,ti=evaluate(model,test_rows,"B1C_selected_test",True);pd.DataFrame([ds]).to_csv(OUT/"dev_summary.csv",index=False);di.to_csv(OUT/"dev_image_metrics.csv",index=False);pd.DataFrame([ts]).to_csv(OUT/"test_summary.csv",index=False);ti.to_csv(OUT/"test_image_metrics.csv",index=False);meta={"gpu":torch.cuda.get_device_name(0),"torch":torch.__version__,"cuda":torch.version.cuda,"chosen_config":spec,"search_results":search,"train_info":info,"train_images":len(train_rows),"dev_images":len(dev),"test_images":len(test_rows),"protocol":f"train_batch={CFG[\'train_batch_size\']}; eval_batch={CFG[\'eval_batch_size\']}; max_side={CFG[\'max_side\']}; EMA"};(OUT/"metadata.json").write_text(json.dumps(meta,indent=2,default=str));print(pd.DataFrame([ts]).to_string(index=False))\nif __name__=="__main__":main()\n').strip() + "\n"
ast.parse(TRAINER_SOURCE)
print("Phase 2.1 trainer syntax: PASS")

In [ ]:
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
run_python("phase21_corrective_runner", TRAINER_SOURCE)

## 5. Report and Gate G2.1

Set `P1_BASELINE_ACCEPTED=True` only after the Phase 1 mismatch has been explained or accepted, and `ARTIFACTS_REVIEWED=True` only after inspecting `failure_cases/`.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown,display
out=OUTPUT_ROOT;p1=pd.read_csv(P1_SUMMARY_PATH).iloc[0];p2=pd.read_csv(P2_SUMMARY_PATH).iloc[0];test=pd.read_csv(out/"test_summary.csv").iloc[0];dev=pd.read_csv(out/"dev_summary.csv").iloc[0];search=pd.read_csv(out/"search_results.csv")
delta_p1=float(test.PSNR-p1.PSNR);speedup=float(p1.original_forward_ms_per_image/test.latency_ms_per_image);gain_p2=float(test.PSNR-p2.PSNR);protocol_ok=int(CONFIG["eval_batch_size"])==int(p1.configured_batch_size) and int(CONFIG["max_side"])==int(p1.max_side);quality_ok=delta_p1>=-.3;speed_ok=speedup>=1.25;corrective_ok=gain_p2>=3.0;artifact_ok=bool(ARTIFACTS_REVIEWED);baseline_ok=bool(P1_BASELINE_ACCEPTED);gate="PASS" if protocol_ok and quality_ok and speed_ok and corrective_ok and artifact_ok and baseline_ok else "REVIEW"
plt.figure(figsize=(7,5));plt.scatter(search.latency_ms_per_image,search.PSNR,s=70);[plt.annotate(n,(x,y)) for n,x,y in zip(search.model,search.latency_ms_per_image,search.PSNR)];plt.axhline(float(p1.PSNR)-.3,color="red",ls="--",label="G2 quality floor");plt.xlabel("Latency (ms/image)");plt.ylabel("PSNR (dB)");plt.legend();plt.grid(alpha=.25);plt.tight_layout();plt.savefig(out/"pareto_quality_speed.png",dpi=160);plt.close()
meta=json.loads((out/"metadata.json").read_text());meta["gate_g2_1"]={"status":gate,"delta_psnr_vs_p1":delta_p1,"speedup_vs_p1":speedup,"psnr_gain_vs_p2":gain_p2,"protocol_ok":protocol_ok,"quality_ok":quality_ok,"speed_ok":speed_ok,"corrective_ok":corrective_ok,"artifact_reviewed":artifact_ok,"p1_baseline_accepted":baseline_ok};(out/"metadata.json").write_text(json.dumps(meta,indent=2,default=str))
lines=["# P2.1 Corrective RGB Report","","## Gate G2.1","",f"Status: {gate}","","| Check | Observed | Criterion | Result |","|---|---:|---:|---|",f"| Protocol match | {protocol_ok} | Phase 1 batch/max_side | {'PASS' if protocol_ok else 'REVIEW'} |",f"| Delta PSNR vs Phase 1 | {delta_p1:.4f} dB | >= -0.3 dB | {'PASS' if quality_ok else 'REVIEW'} |",f"| Speedup vs Phase 1 | {speedup:.4f}x | >= 1.25x | {'PASS' if speed_ok else 'REVIEW'} |",f"| PSNR gain vs failed Phase 2 | {gain_p2:.4f} dB | >= 3.0 dB | {'PASS' if corrective_ok else 'REVIEW'} |",f"| Artifact review | {artifact_ok} | complete | {'PASS' if artifact_ok else 'REVIEW'} |",f"| Phase 1 baseline accepted | {baseline_ok} | mismatch explained | {'PASS' if baseline_ok else 'REVIEW'} |","","## Selected model","",f"- Dev PSNR/SSIM: {dev.PSNR:.4f} / {dev.SSIM:.6f}",f"- Test PSNR/SSIM: {test.PSNR:.4f} / {test.SSIM:.6f}",f"- Latency/FPS: {test.latency_ms_per_image:.4f} ms / {test.FPS:.2f}",f"- Parameters: {test.params_M:.4f} M",f"- Peak VRAM: {test.peak_allocated_GiB:.4f} GiB","","## Decision","","Proceed to Phase 3 only if G2.1 is PASS. If quality remains below the threshold, retain/fine-tune part of the original DEN instead of adding wavelet or ASM branches."]
(out/"P2_1_corrective_report.md").write_text("\n".join(lines)+"\n");display(Markdown((out/"P2_1_corrective_report.md").read_text()))

## 6. Package Phase 2.1 results

In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive(str(WORK_ROOT/"wdmamba_phase21_results"),"zip",OUTPUT_ROOT);print("Created:",archive);files.download(archive)